Middleware is software that acts as an intermediary between different applications, services, or systems. It facilitates communication, data management, and integration by providing common services and capabilities. Middleware can be used in various contexts, such as web development, enterprise applications, and distributed systems.

objectives of middleware in agents are
1. **Communication Facilitation**: Middleware enables seamless communication between different agents, allowing them to exchange messages and data efficiently.
2. **Interoperability**: Middleware provides a common interface and protocols that allow agents developed in different programming languages or platforms to work together without compatibility issues.
3. **Resource Management**: Middleware helps manage resources such as memory, processing power, and network bandwidth, ensuring that agents can operate effectively without resource contention.
4. **Scalability**: Middleware supports the scaling of agent systems by allowing new agents to be added or removed dynamically, ensuring that the system can handle increased load or changing requirements.
5. **Security**: Middleware can provide security features such as authentication, authorization, and encryption to protect the communication and data exchanged between agents, ensuring that sensitive information is safeguarded. 


In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is missing. Add it to your environment or .env file before running the notebook.")


### Summarization Middleware 
Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older
context. Summarization is useful for the following:

. Long-running conversations that exceed context windows.
· Multi-turn dialogues with extensive history.
. Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
from langchain.chat_models import init_chat_model

model=init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq"
)
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [ ]:
config={"configurable":{"thread_id":"test-1"}}

In [ ]:

questions = [
"What is 2+2?",
"What is 10*5?",
"What is 100/4?",
"What is 15-7?",
"What is 3+3?",
"What is 4*4?", 
]

In [ ]:
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config=config)
    print(response)
    print(len(response['messages']))

### token size

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens. """
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

In [ ]:
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",500),
            keep=("tokens",200)
        )
    ]
)

In [ ]:
def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars//4

In [ ]:
questions = [
    "bhubaneswar",
    "delhi",
    
]

config = {
    "configurable": {
        "thread_id": "1"
    }
}

for q in questions:

    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"find hotels in {q}")
            ]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])

    print(f"{q} : ~{tokens} tokens, {len(response['messages'])} messages")
    print(response["messages"])
    print("-" * 50)

# Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the
following:

. High-stakes operations requiring human approval (e.g. database writes, financial transactions).
. Compliance workflows where human oversight is mandatory.
. Long-running conversations where human feedback guides the agent.

In [125]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

@tool 
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"""

In [126]:
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]

                },
                "read_email_tool":False
            }
        )

    ]
)

In [127]:

loop=True
config={"configurable":{"thread_id":"test-approve6"}}

result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to jhon@test.com with subject 'hello' and body 'How are you?'" )]}
    ,config=config
)



In [128]:
result



{'messages': [HumanMessage(content="Send email to jhon@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='431b848a-728d-422e-a215-e4ee272b32c9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the send_email_tool. Provide JSON.', 'tool_calls': [{'id': 'fc_a7fa6467-74ca-4812-8d52-8f46f1ffc36f', 'function': {'arguments': '{"body":"How are you?","recipient":"jhon@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 175, 'total_tokens': 226, 'completion_time': 0.05285447, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.008632826, 'prompt_tokens_details': None, 'queue_time': 0.342925771, 'total_time': 0.061487296}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9b8528b477', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provid

In [129]:
from langgraph.types import Command

In [130]:
if "__interrupt__" in result:
    print("Paused! What do you want to do?")
    user_response = input("approve / edit / reject: ").strip().lower()

    if user_response != "edit":
        result = agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {"type": user_response}
                    ]
                }
            ),
            config=config
        )

    else:
        result = agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            "type": "edit",
                            "edited_action": {
                                "name": "send_email_tool",
                                "args": {
                                    "recipient": "correct@gmail.com",
                                    "subject": "corrected Subject",
                                    "body": "This was edited by human before sending"
                                }
                            }
                        }
                    ]
                }
            ),
            config=config
        )

Paused! What do you want to do?


In [131]:
result

{'messages': [HumanMessage(content="Send email to jhon@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='431b848a-728d-422e-a215-e4ee272b32c9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the send_email_tool. Provide JSON.', 'tool_calls': [{'id': 'fc_a7fa6467-74ca-4812-8d52-8f46f1ffc36f', 'function': {'arguments': '{"body":"How are you?","recipient":"jhon@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 175, 'total_tokens': 226, 'completion_time': 0.05285447, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.008632826, 'prompt_tokens_details': None, 'queue_time': 0.342925771, 'total_time': 0.061487296}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9b8528b477', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provid

In [132]:
result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config=config
)

In [133]:
from importlib.metadata import version

print("langchain:", version("langchain"))
print("langgraph:", version("langgraph"))
print("langgraph-prebuilt:", version("langgraph-prebuilt"))

langchain: 1.4.2
langgraph: 1.2.12
langgraph-prebuilt: 1.1.0


In [134]:
result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config=config
)

In [135]:
result

{'messages': [HumanMessage(content="Send email to jhon@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='431b848a-728d-422e-a215-e4ee272b32c9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the send_email_tool. Provide JSON.', 'tool_calls': [{'id': 'fc_a7fa6467-74ca-4812-8d52-8f46f1ffc36f', 'function': {'arguments': '{"body":"How are you?","recipient":"jhon@test.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 175, 'total_tokens': 226, 'completion_time': 0.05285447, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.008632826, 'prompt_tokens_details': None, 'queue_time': 0.342925771, 'total_time': 0.061487296}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9b8528b477', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provid